In [ ]:
import os
import sys
import tempfile
import pandas as pd
import ray
import scanpy as sc
import scvi
import mudata as md
import muon
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from ray import tune
from scvi import autotune
import celltypist
from sklearn_ann.kneighbors.annoy import AnnoyTransformer
print("Last run with scvi-tools version:", scvi.__version__)
os.environ["TUNE_DISABLE_STRICT_METRIC_CHECKING"] = "1"

In [ ]:
scvi.settings.num_threads = 24
scvi.settings.seed = 0

sc._settings.ScanpyConfig.n_jobs=4
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,frameon=False,
    facecolor = 'white', figsize=(8,8), format='png')

sns.set_theme()
#torch.set_float32_matmul_precision("high")

# 0. Loading and preprocessing the dataset

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7"

obj_path = '/home/liyanguo/MyImmuCell/04_MyImmuCell_TOTALVI/'
sc.settings.figdir = obj_path
dataset = sys.argv[1] #Low or high
#dataset = 'high_nCount_RNA'
adata = sc.read_h5ad(f"{obj_path}scRNA_MyImmuCell_{dataset}_HVG.h5ad",backed='r')
adt = sc.read_h5ad(f"{obj_path}scADT_MyImmuCell_{dataset}.h5ad",backed='r')

# downsample 5000 cell in each cell type
ncells = int(sys.argv[2])
#ncells = 5000
cell_index = celltypist.samples.downsample_adata(adata,mode = 'each', n_cells = ncells,
                                                 by = 'Reference_Atlas_L1L2_pl',
                                                 return_index = True,random_state=0)

adt = adt[cell_index].to_memory()

adata = adata[cell_index].to_memory()

# 1. Model hyperparameter tuning with TOTALVI

In [ ]:
adata.obsm["protein"] = adt.X.copy()
adata.X = adata.layers['counts'].copy()
del adata.layers

In [ ]:
adata.obs['Batch'] = adt.obs['Batch']

In [ ]:
adata

In [ ]:
adata.write(f"{obj_path}Model_hyperparameter_TOTALVI_MyImmuCell_{dataset}.h5ad",compression="gzip")

In [ ]:
search_space = {
    "model_params": {"n_layers_decoder": tune.choice([1, 2]),"n_hidden": tune.choice([128, 256]),"n_latent": tune.choice([20, 30])},
    "train_params": {"max_epochs": 150, 'batch_size':tune.choice([256, 512, 1024,4096]), "lr": tune.choice([1e-5,1e-4,5e-4,1e-3,4e-3,1e-2])},
}

In [ ]:
ray.shutdown()

In [ ]:
model_cls = scvi.model.TOTALVI

model_cls.setup_anndata(adata,
                        protein_expression_obsm_key='protein',
                        batch_key="Batch"
                       )

results = autotune.run_autotune(
    model_cls,
    data=adata,
    mode="min",
    seed=0,
    metrics=["validation_loss",'train_loss_step','train_loss_epoch','elbo_train','elbo_validation','reconstruction_loss_validation','reconstruction_loss_train'],
    search_space=search_space,
    num_samples=90,
    resources={"cpu": 24, "gpu": 2},
)

print(results.result_grid)

model_cls.save(obj_path, overwrite=True, prefix=f'{dataset}_CITEseq_TOTALVI_')

In [ ]:
ray.shutdown()

In [ ]:
# Run in Shell
nohup python 08A_Model_hyperparameter_tuning_with_TOTALVI_gpu6.py high_nCount_RNA 5000 > high_nCount_RNA_tuning.log 2>&1 &
nohup python 08A_Model_hyperparameter_tuning_with_TOTALVI_gpu7.py low_nCount_RNA 10000 > low_nCount_RNA_tuning.log 2>&1 &

# Model hyperparameter tuning results

In [ ]:
restored_tuner = tune.Tuner.restore("/data/lyg/MyImmuCell/scvi_log/totalvi_eaff608e-c244-4e55-91fd-8821f1b2740f/totalvi_eaff608e-c244-4e55-91fd-8821f1b2740f", trainable=train_mnist)

In [ ]:
print(result_grid.get_best_result())

In [ ]:
# large batch_size get larger validation_loss
# batch_size 256, lr 0.01, and n_layers_decoder 1, get smallest validation_loss and similar train_loss
# n_layers_decoder 

In [ ]:
# 选择

# 2. CITE-seq analysis with totalVI

In [ ]:
mdata = md.MuData({"rna": adata, "protein": adt})
mdata.update()

In [ ]:
print(f"Max value of protein counts that store in X: {mdata.mod['protein'].X.max()}")

In [ ]:
print(f"Max value of protein counts that store in layers-counts: {mdata.mod['rna'].layers['counts'].max()}")

In [ ]:
scvi.model.TOTALVI.setup_mudata(
    mdata,
    batch_key="Batch",
    modalities={
        "rna_layer": "rna",
        "protein_layer": "protein",
        "batch_key": "protein",
    },
)

In [ ]:
model = scvi.model.TOTALVI(mdata, n_layers_decoder=3)

In [ ]:
model.train(accelerator="auto",lr=0.004,
            max_epochs=400,batch_size=1024)

# 3. Get_latent_representation

In [ ]:
adata.obsm["X_TOTALVI"] = model.get_latent_representation()

# 4. Get model parameter

In [ ]:
sheet=pd.DataFrame()
for key in model.history.keys():
    temp = model.history[key]
    temp = temp.reset_index()
    sheet=pd.concat([sheet,temp],axis=1)

In [ ]:
pd.DataFrame.to_csv(sheet,f"{obj_path}{dataset}_TOTALVI_model_history.csv")

In [ ]:
fig, ax = plt.subplots(1, 1)
sheet["elbo_train"].plot(ax=ax, label="train")
sheet["elbo_validation"].plot(ax=ax, label="validation")
ax.set(title="Negative ELBO over training epochs", ylim=(0, 1400))
ax.legend()
plt.savefig(f"{obj_path}{dataset}_ELBO.png")

# 5. Cluster

In [ ]:
%%time
sc.pp.neighbors(adata, transformer=AnnoyTransformer(20), use_rep='X_TOTALVI')

In [ ]:
%%time
sc.tl.umap(adata, min_dist=0.5)

In [ ]:
%%time
res = 3
sc.tl.leiden(adata, key_added=f'L1_leiden_TOTALVI_{res}', resolution=res,
             use_weights=True,directed=False,flavor="igraph")

In [ ]:
sc.pl.umap(
    adata,
    color=['L1_leiden_TOTALVI_3'],
    legend_fontsize=6,legend_loc='on data',ncols=3,frameon=False,
    save=f'_{dataset}_TOTALVI_L1_annoy_leiden'
)